In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Create Synthetic Resume Data
resume_data = {
    'Candidate_Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Resume_Text': [
        "Experienced Python developer with strong skills in machine learning, NLP, scikit-learn, and data analysis.",
        "Frontend web developer specializing in HTML, CSS, JavaScript, and React framework.",
        "Data Scientist proficient in Python, SQL, Tableau, and predictive modeling using random forest.",
        "Recent graduate with basic knowledge of C++ and Java. Looking for an entry-level software role."
    ]
}
df_resumes = pd.DataFrame(resume_data)

# 2. Define the target Job Description
job_description = "We are looking for a Machine Learning Engineer with expertise in Python, NLP, scikit-learn, and data science."

print("Resume Dataset and Job Description Created Successfully!")
display(df_resumes)

Resume Dataset and Job Description Created Successfully!


,Candidate_Name,Resume_Text
0,Alice,Experienced Python developer with strong skill...
1,Bob,"Frontend web developer specializing in HTML, C..."
2,Charlie,"Data Scientist proficient in Python, SQL, Tabl..."
3,David,Recent graduate with basic knowledge of C++ an...


In [2]:
# Set up our stop words
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Tokenize and remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

# Clean both the resumes and the job description
df_resumes['Cleaned_Resume'] = df_resumes['Resume_Text'].apply(clean_text)
cleaned_jd = clean_text(job_description)

print("Text cleaning complete!")
print(f"Cleaned Target Job: '{cleaned_jd}'")

Text cleaning complete!
Cleaned Target Job: 'looking machine learning engineer expertise python nlp scikitlearn data science'


In [3]:
# Initialize the Vectorizer
vectorizer = TfidfVectorizer()

# Combine all text to teach the AI the vocabulary
all_text = df_resumes['Cleaned_Resume'].tolist() + [cleaned_jd]
tfidf_matrix = vectorizer.fit_transform(all_text)

# Separate the resumes from the job description
resume_vectors = tfidf_matrix[:-1] 
jd_vector = tfidf_matrix[-1] 

# Calculate the Match Score (Cosine Similarity)
similarities = cosine_similarity(resume_vectors, jd_vector).flatten()

# Convert scores to percentages and rank the candidates
df_resumes['Match_Score (%)'] = np.round(similarities * 100, 2)
df_ranked = df_resumes.sort_values(by='Match_Score (%)', ascending=False).reset_index(drop=True)

print("Candidate Screening and Ranking Complete!\n")
display(df_ranked[['Candidate_Name', 'Match_Score (%)', 'Resume_Text']])

Candidate Screening and Ranking Complete!



,Candidate_Name,Match_Score (%),Resume_Text
0,Alice,45.85,Experienced Python developer with strong skill...
1,Charlie,10.66,"Data Scientist proficient in Python, SQL, Tabl..."
2,David,8.28,Recent graduate with basic knowledge of C++ an...
3,Bob,0.00,"Frontend web developer specializing in HTML, C..."


In [4]:
# Extract target skills from the Job Description for Skill Gap Identification
target_skills = ['python', 'machine learning', 'nlp', 'scikit-learn', 'data science']

def identify_missing_skills(text):
    text_lower = text.lower()
    # Check which target skills are NOT in the candidate's resume
    missing = [skill for skill in target_skills if skill not in text_lower]
    return ", ".join(missing) if missing else "None (Fully Qualified)"

# Apply the skill gap identifier to our ranked candidates
df_ranked['Missing_Skills'] = df_ranked['Resume_Text'].apply(identify_missing_skills)

print("Skill Gap Identification Complete!\n")
display(df_ranked[['Candidate_Name', 'Match_Score (%)', 'Missing_Skills']])

Skill Gap Identification Complete!



,Candidate_Name,Match_Score (%),Missing_Skills
0,Alice,45.85,data science
1,Charlie,10.66,"machine learning, nlp, scikit-learn, data science"
2,David,8.28,"python, machine learning, nlp, scikit-learn, d..."
3,Bob,0.00,"python, machine learning, nlp, scikit-learn, d..."


### Business Insights & HR System Explanation

**How Resumes are Scored & Ranked:**
This system uses Natural Language Processing (NLP) to convert the text of a candidate's resume and the target Job Description into mathematical vectors (TF-IDF). It then calculates the "Cosine Similarity" between them. A higher percentage means the candidate's experience closely aligns with the job requirements. Candidates are then ranked from highest score to lowest.

**Skill Gap Identification (Why Candidates Rank Higher):**
* Candidates like Alice rank at the top because their resume contains the exact keyword matches for the required ML stack.
* Candidates who rank lower are explicitly flagged in the `Missing_Skills` column. 

**For an HR Manager or Recruiter, this means:**
* **Massive Time Savings:** Hundreds of resumes can be instantly sorted so human recruiters only spend time reading the top 10% of matches.
* **Objective Filtering:** Reduces human bias in the initial screening phase by strictly evaluating technical skill overlap.
* **Targeted Interviews:** Recruiters can look at the "Missing Skills" column to know exactly what technical gaps to ask the candidate about during the phone screen.